# IR (HIT-UAV) - 01 Explorar dataset

Que hace: descomprime HIT-UAV desde Drive, indexa imagenes y labels, y responde las preguntas abiertas del dataset termico:
formato de labels, indice de `DontCare`, si las imagenes son gris, balance de clases y tamano de los objetos.

Que NO hace: no carga ni entrena ningun modelo (eso es `02_entrenar`).

## Parametros

In [ ]:
DATASET_ZIP = "MyDrive/deteccion_drones/datasets/hituav.zip"   # zip bajado desde la web de Kaggle y subido a Drive
DATASET_DIR = "hituav"                                  # subcarpeta en env.DATA_ROOT
DATA_YAML   = "hit-uav/dataset.yaml"                    # relativo a DATASET_DIR. VERIFICAR con la celda de la seccion 1

## Setup

Clona/actualiza el repo, instala el paquete y monta Drive. Identica en todos los notebooks (salvo el import: 01 y 00 no necesitan `experiment.py`).

Requiere que el repo sea publico (un token via Colab Secrets no es confiable desde la extension de Colab para VS Code: `userdata.get` depende del frontend web).

In [ ]:
REPO = "https://github.com/sbstn-sss/deteccion_drones.git"
REPO_DIR = "/content/deteccion_drones"
BRANCH = "feat/framework"   # cambiar para probar una rama antes del merge, ej "feat/framework"

import sys
if "google.colab" in sys.modules:
    !git -C {REPO_DIR} checkout -q {BRANCH} 2>/dev/null && git -C {REPO_DIR} pull -q 2>/dev/null || git clone -q -b {BRANCH} {REPO} {REPO_DIR}
    !pip install -q -e {REPO_DIR}
    sys.path.insert(0, f"{REPO_DIR}/src")   # pip -e usa un .pth que el kernel ya iniciado no lee: sin esto falla el import

try:  # el IPython de Colab con Python 3.13 trae un autoreload que importa 'imp' (eliminado en 3.12)
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except ModuleNotFoundError:
    print("autoreload no disponible: reinicia el kernel tras editar src/")
from deteccion import env, data, viz
env.mount_drive()

## 1. Obtener el dataset

Zip en Drive en vez de la API de Kaggle: la API necesita credenciales en los Secrets de Colab, y `userdata.get`
no es confiable desde la extension de VS Code. `unzip_once` no re-descomprime si ya termino (marcador `.done`).

La ultima linea lista los `.yaml` que trae el zip: si `DATA_YAML` no coincide, corrige el parametro y vuelve a correr.

In [ ]:
import pandas as pd

dataset_dir = env.DATA_ROOT / DATASET_DIR
env.unzip_once(env.drive_path(DATASET_ZIP), dataset_dir)
sorted(str(p.relative_to(dataset_dir)) for p in dataset_dir.rglob("*.yaml"))

El yaml corregido muestra los nombres de clase con su indice (ahi se ve el indice de `DontCare`) y las carpetas de cada split.

In [ ]:
data_yaml = env.fix_data_yaml(dataset_dir / DATA_YAML)
print(data_yaml.read_text())

## 2. Indexar imagenes y labels

Si casi todas las lineas terminan en `issues` con reason `n_cols`, las labels **no** estan en formato YOLO
(HIT-UAV tambien se distribuye en VOC/COCO): parar aqui, hara falta un converter.

In [ ]:
d = data.load_data_yaml(data_yaml)
splits = [s for s in ("train", "val", "test") if d["splits"][s] is not None]
images = pd.concat([data.index_split(d, s) for s in splits], ignore_index=True)
boxes, issues = data.load_boxes(images, d["names"])

## 3. Resumen y lineas invalidas

In [ ]:
data.summary(images, boxes, issues)

In [ ]:
issues["reason"].value_counts()

## 4. Distribucion de clases y `DontCare`

`DontCare` marca zonas ambiguas, no objetos. Si se entrena como clase, el modelo aprende a "detectar" regiones confusas.
Opcion por defecto: excluirla en `02_entrenar` con `TRAIN_ARGS["classes"] = [indices sin DontCare]`. Decidir con estos conteos.

In [ ]:
data.class_counts(boxes)

In [ ]:
viz.plot_class_counts(boxes, d["names"])

## 5. Canales de imagen

Se espera gris: 1 canal, o 3 canales identicos (JPG guardado como RGB). En ambos casos YOLO lo usa tal cual, sin convertir.
Si aparece "color", las imagenes traen una paleta (ironbow, etc.) y eso importa para la camara real.

In [ ]:
import numpy as np
from PIL import Image

def canales(p):
    a = np.asarray(Image.open(p))
    if a.ndim == 2:
        return "1 canal (gris)"
    return "3 canales iguales (gris)" if (a[..., 0] == a[..., 1]).all() and (a[..., 1] == a[..., 2]).all() else "color"

images.sample(50, random_state=0)["image_path"].map(canales).value_counts()

## 6. Tamanos de imagen y de caja

HIT-UAV es chico (~2.9k imagenes, 640x512 segun el paper): se leen todos los headers, sin `sample`.

In [ ]:
images = data.add_image_sizes(images)
viz.plot_image_sizes(images)

In [ ]:
boxes_px = data.add_pixel_sizes(boxes, images)
viz.plot_box_sizes(boxes_px)

In [ ]:
pd.crosstab(boxes_px["name"], boxes_px["size_bucket"])

## 7. Cajas por imagen

In [ ]:
viz.plot_boxes_per_image(boxes, images)

## 8. Muestras con ground truth

In [ ]:
viz.show_samples(images, boxes, d["names"], n=6, seed=0)

## Conclusiones

_(completar tras revisar)_
- Labels en YOLO: si / no
- Indice de `DontCare` y decision (excluir / mantener):
- Imagenes gris: si / no
- % de `Person` small -> imgsz para `02_entrenar`:
- Balance Person vs Car (prioridad: personas):